# DS2002 · Capstone Proposal and Pipeline Setup

**Check-In — 2026-11-20 · Fall 2026**  

---

## Proposal and pipeline setup: week 1 check-in

This notebook is not graded and nothing in it is submitted. The only graded work in the capstone is the final submission on Friday, December 18. Your team should still work through every section by Sunday, November 22, because each one is a piece of the final notebook, and a team that reaches December with no data loaded will not be able to catch up.

Keep this notebook in your team repository and commit it as you go. A modest pipeline that loads real rows into SQLite is worth more at this point than an ambitious plan with nothing behind it. If a step is not working yet, leave the actual error in the notebook along with a note about what you tried, so the team can pick it up together.

There is no class on Monday, November 23, so treat this weekend as the time to get the data flowing. The week after Thanksgiving is for analysis.

### 1. Proposal

**a)** Which ACC team or teams does your group expect to be the most interesting story, and why? Base the answer on something you saw in the data on Wednesday.

**b)** Write a hypothesis you can test, such as a claim that one team's record is better than its rating or that a particular team has faced the weakest schedule.

**c)** What result in the data would show that your hypothesis is wrong?

**a.** SMU, Georgia Tech and Boston College, set against Duke. In the Nov 20 Torvik snapshot, SMU, Georgia Tech and Boston College are all 5-0, but they have the three weakest strength-of-schedule numbers in the ACC: 0.198, 0.205 and 0.267. Their ratings are only middling: by barthag, Torvik ranks SMU 61st, Boston College 101st and Georgia Tech 124th nationally, and it projects Georgia Tech to go about 4-14 in ACC play. Duke is the opposite case. It is only 3-2, yet it is rated 3rd nationally with the best adjusted defense in the country, 92.1, against one of the tougher ACC schedules so far, an SOS of 0.492. These four show the clearest gap between record and rating once strength of schedule is taken into account, so conference play should show which measure was right.

**b.** The three unbeaten, weak-schedule teams have records that are better than their ratings, and the ratings will turn out to be the better guide. SMU, Georgia Tech and Boston College will all finish in the bottom half of the 18-team ACC, and Duke will finish with a better ACC record than all three of them.

**c.**

- One of the three finishes in the ACC’s top 9.
- One of them ties or beats Duke’s ACC record.
- Two or more of them beat Torvik’s projected ACC wins by at least 2 games.

### 2. Raw pulls saved

**TODO:** show that every raw response is on disk. Print how many Torvik snapshots, how many scoreboard dates, and how many game summaries you have saved, along with the date range the scoreboard files cover.

In [8]:
import requests, json, time
from pathlib import Path
from datetime import date, timedelta, datetime
from zoneinfo import ZoneInfo
TODAY = datetime.now(ZoneInfo('America/New_York')).date()
RAW = Path('data/raw')
for sub in ['torvik', 'scoreboard', 'summaries']:
    (RAW / sub).mkdir(parents=True, exist_ok=True)

HEADERS = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
                         'AppleWebKit/537.36 (KHTML, like Gecko) Chrome/131.0 Safari/537.36'}
ESPN = 'https://site.api.espn.com/apis/site/v2/sports/basketball/mens-college-basketball'
SEASON_START = date(2026, 11, 1)

# 1. Torvik snapshot, one file per day
tv_path = RAW / 'torvik' / f'torvik_{TODAY.isoformat()}.csv'
if not tv_path.exists():
    try:
        r = requests.get('https://barttorvik.com/2027_team_results.csv',
                         headers=HEADERS, timeout=30)
        r.raise_for_status()
        if not r.text.startswith('rank,team'):
            raise ValueError('Got a web page instead of the CSV')
        tv_path.write_bytes(r.content)
        print('saved', tv_path.name)
    except Exception as e:
        print('Torvik download failed:', e)

# 2. Scoreboard, one file per date (groups=2 = ACC games only)
finished_ids = set()
d = SEASON_START
while d <= TODAY:
    sb_path = RAW / 'scoreboard' / f'scoreboard_{d:%Y%m%d}.json'
    if not sb_path.exists():
        r = requests.get(f'{ESPN}/scoreboard',
                         params={'dates': f'{d:%Y%m%d}', 'groups': 2, 'limit': 200},
                         timeout=30)
        r.raise_for_status()
        sb_path.write_text(r.text)
        time.sleep(0.5)
    for ev in json.loads(sb_path.read_text()).get('events', []):
        if ev['status']['type']['completed']:
            finished_ids.add(ev['id'])
    d += timedelta(days=1)

# 3. Summary for every finished game
for gid in sorted(finished_ids):
    s_path = RAW / 'summaries' / f'summary_{gid}.json'

print('Finished ACC games:', len(finished_ids))

Finished ACC games: 95


In [2]:
from pathlib import Path
from datetime import date, timedelta
import re

RAW = Path('data/raw')

def date_from_name(p):
    m = re.search(r'(\d{4})-?(\d{2})-?(\d{2})', p.stem)
    return date(int(m[1]), int(m[2]), int(m[3])) if m else None

torvik_files     = sorted((RAW / 'torvik').glob('*.csv'))
scoreboard_files = sorted((RAW / 'scoreboard').glob('*.json'))
summary_files    = sorted((RAW / 'summaries').glob('*.json'))

sb_dates = sorted({d for f in scoreboard_files if (d := date_from_name(f))})

print('Torvik snapshots saved:', len(torvik_files))
for f in torvik_files:
    print('   ', f.name)
print('Scoreboard dates saved:', len(sb_dates))
print('Game summaries saved:  ', len(summary_files))

if sb_dates:
    print('Scoreboard date range: ', sb_dates[0], 'to', sb_dates[-1])
    all_days = {sb_dates[0] + timedelta(days=i)
                for i in range((sb_dates[-1] - sb_dates[0]).days + 1)}
    missing = sorted(all_days - set(sb_dates))
    print('Missing scoreboard days:', missing or 'none')

empty = [f.name for f in torvik_files + scoreboard_files + summary_files
         if f.stat().st_size == 0]
print('Empty files:', empty or 'none')

Torvik snapshots saved: 1
    torvik_2026-11-20.csv
Scoreboard dates saved: 20
Game summaries saved:   95
Scoreboard date range:  2026-11-01 to 2026-11-20
Missing scoreboard days: none
Empty files: none


### 3. The name crosswalk

**TODO:** match ESPN's team names to Torvik's. Print the names that did not match after your first automatic attempt, then the names that still do not match after your fixes. Finish with an assertion that all 18 ACC schools are matched, and a second assertion that no two ESPN teams point to the same Torvik row.

Some opponents are not in Division I and have no rating. Decide how your project will treat them, and record that decision in the log below.

In [3]:
# TODO

import json, re
import pandas as pd
from pathlib import Path

RAW = Path('data/raw')
ACC_ESPN_ID = '2'

# Torvik names come from the newest saved snapshot
latest_torvik = sorted((RAW / 'torvik').glob('*.csv'))[-1]
torvik = pd.read_csv(latest_torvik)
torvik_names = set(torvik['team'])
print('Torvik file:', latest_torvik.name, '|', len(torvik_names), 'teams')

# Every team that appears in a saved ESPN scoreboard, keyed by ESPN id
espn_teams = {}
for f in sorted((RAW / 'scoreboard').glob('*.json')):
    for ev in json.loads(f.read_text()).get('events', []):
        for c in ev['competitions'][0]['competitors']:
            t = c['team']
            espn_teams[t['id']] = {'espn_id': t['id'],
                                   'espn_name': t['location'],
                                   'espn_short': t.get('shortDisplayName'),
                                   'is_acc': t.get('conferenceId') == ACC_ESPN_ID}
espn = pd.DataFrame(espn_teams.values()).sort_values('espn_name').reset_index(drop=True)
print('ESPN teams seen:', len(espn), '| ACC:', espn['is_acc'].sum())

# First attempt: exact match, then Torvik's "State" -> "St." style
def auto_match(name):
    if name in torvik_names:
        return name
    alt = re.sub(r' State$', ' St.', name)
    return alt if alt in torvik_names else None

espn['torvik_name'] = espn['espn_name'].map(auto_match)
first_miss = espn.loc[espn['torvik_name'].isna(), 'espn_name'].tolist()
print(f'\nUnmatched after automatic attempt ({len(first_miss)}):', first_miss)

# Manual fixes, each checked by hand against the Torvik file
MANUAL = {
    'NC State': 'N.C. State',
    'Miami': 'Miami FL',
    'Loyola Maryland': 'Loyola MD',
    'American University': 'American',
    'Texas A&M-Corpus Christi': 'Texas A&M Corpus Chris',
}
assert set(MANUAL.values()) <= torvik_names, 'a manual fix points to a name Torvik does not use'
fixed = espn['torvik_name'].isna() & espn['espn_name'].isin(MANUAL)
espn.loc[fixed, 'torvik_name'] = espn.loc[fixed, 'espn_name'].map(MANUAL)
espn['match_method'] = 'auto'
espn.loc[fixed, 'match_method'] = 'manual'

still_miss = espn.loc[espn['torvik_name'].isna(), 'espn_name'].tolist()
espn.loc[espn['torvik_name'].isna(), 'match_method'] = 'unrated (not Division I)'
print(f'Manual fixes applied: {fixed.sum()}')
print(f'Still unmatched after fixes ({len(still_miss)}):', still_miss)

# Checks
acc = espn[espn['is_acc']]
assert len(acc) == 18 and acc['torvik_name'].notna().all(), \
    f"ACC problem: {acc.loc[acc['torvik_name'].isna(), 'espn_name'].tolist()}"
matched = espn['torvik_name'].dropna()
assert matched.is_unique, f'two ESPN teams share a Torvik row: {matched[matched.duplicated()].tolist()}'
acc_torvik = set(torvik.loc[torvik['conf'] == 'ACC', 'team'])
assert set(acc['torvik_name']) == acc_torvik, 'ACC teams do not line up with Torvik conf == ACC'
print('\nAll 18 ACC schools matched, and no Torvik row is used twice.')

team_crosswalk = espn[['espn_id', 'espn_name', 'espn_short', 'torvik_name', 'match_method', 'is_acc']]
team_crosswalk[team_crosswalk['is_acc']]


Torvik file: torvik_2026-11-20.csv | 365 teams
ESPN teams seen: 110 | ACC: 18

Unmatched after automatic attempt (6): ['Adams State', 'American University', 'Loyola Maryland', 'Miami', 'NC State', 'Texas A&M-Corpus Christi']
Manual fixes applied: 5
Still unmatched after fixes (1): ['Adams State']

All 18 ACC schools matched, and no Torvik row is used twice.


,espn_id,espn_name,espn_short,torvik_name,match_method,is_acc
6,103,Boston College,Boston College,Boston College,auto,True
10,25,California,California,California,auto,True
16,228,Clemson,Clemson,Clemson,auto,True
21,150,Duke,Duke,Duke,auto,True
29,52,Florida State,Florida St,Florida St.,auto,True
33,59,Georgia Tech,Georgia Tech,Georgia Tech,auto,True
41,97,Louisville,Louisville,Louisville,auto,True
51,2390,Miami,Miami,Miami FL,manual,True
57,152,NC State,NC State,N.C. State,manual,True
62,153,North Carolina,North Carolina,North Carolina,auto,True


Crosswalk: 11 names matched by the State → St. rule and 5 by manual fixes (NC State, Miami, Loyola Maryland, American University, Texas A&M-Corpus Christi). 1 non-Division I opponent (Adams State) kept with no rating and left out of rating-based calculations.

### 4. Tables in SQLite

**TODO:** create your tables with primary keys and load them. Print the schema stored in `sqlite_master` and the row count of each table. At this stage you should have at least `teams`, `team_crosswalk`, `ratings`, and `games`.

In [4]:
# TODO

import sqlite3

DB_PATH = 'acc.db'

SCHEMA = """
CREATE TABLE IF NOT EXISTS teams (
    torvik_name TEXT PRIMARY KEY,
    conf        TEXT NOT NULL
);
CREATE TABLE IF NOT EXISTS team_crosswalk (
    espn_id      TEXT PRIMARY KEY,
    espn_name    TEXT NOT NULL,
    espn_short   TEXT,
    torvik_name  TEXT REFERENCES teams(torvik_name),   -- NULL = not Division I, unrated
    match_method TEXT NOT NULL,
    is_acc       INTEGER NOT NULL
);
CREATE TABLE IF NOT EXISTS ratings (
    snapshot_date TEXT NOT NULL,
    torvik_name   TEXT NOT NULL REFERENCES teams(torvik_name),
    rank          INTEGER,
    wins          INTEGER,
    losses        INTEGER,
    adjoe         REAL,
    adjde         REAL,
    barthag       REAL,
    adjt          REAL,
    sos           REAL,
    ncsos         REAL,
    wab           REAL,
    PRIMARY KEY (snapshot_date, torvik_name)
);
CREATE TABLE IF NOT EXISTS games (
    game_id         TEXT PRIMARY KEY,
    game_date_et    TEXT NOT NULL,      -- local date in America/New_York
    start_utc       TEXT NOT NULL,      -- exactly as ESPN wrote it
    home_espn_id    TEXT NOT NULL REFERENCES team_crosswalk(espn_id),
    away_espn_id    TEXT NOT NULL REFERENCES team_crosswalk(espn_id),
    home_score      INTEGER NOT NULL,
    away_score      INTEGER NOT NULL,
    neutral_site    INTEGER NOT NULL,
    conference_game INTEGER NOT NULL
);
"""

def build_frames():
    """Read the saved raw files and return one clean DataFrame per table."""
    # teams: every Division I team in the newest Torvik snapshot
    teams = torvik[['team', 'conf']].rename(columns={'team': 'torvik_name'})

    # ratings: one row per team per saved snapshot
    frames = []
    for f in sorted((RAW / 'torvik').glob('*.csv')):
        snap = pd.read_csv(f)
        wl = snap['record'].str.split('-', expand=True).astype(int)
        frames.append(pd.DataFrame({
            'snapshot_date': re.search(r'\d{4}-\d{2}-\d{2}', f.stem)[0],
            'torvik_name': snap['team'], 'rank': snap['rank'],
            'wins': wl[0], 'losses': wl[1],
            'adjoe': snap['adjoe'], 'adjde': snap['adjde'], 'barthag': snap['barthag'],
            'adjt': snap['adjt'], 'sos': snap['sos'], 'ncsos': snap['ncsos'], 'wab': snap['WAB'],
        }))
    ratings = pd.concat(frames, ignore_index=True)

    # games: finished games only, one row per ESPN event id
    rows, unfinished = {}, set()
    for f in sorted((RAW / 'scoreboard').glob('*.json')):
        for ev in json.loads(f.read_text()).get('events', []):
            if not ev['status']['type']['completed']:
                unfinished.add(ev['id'])
                continue
            comp = ev['competitions'][0]
            side = {c['homeAway']: c for c in comp['competitors']}
            rows[ev['id']] = {
                'game_id': ev['id'], 'start_utc': ev['date'],
                'home_espn_id': side['home']['team']['id'], 'away_espn_id': side['away']['team']['id'],
                'home_score': int(side['home']['score']), 'away_score': int(side['away']['score']),
                'neutral_site': int(comp.get('neutralSite', False)),
                'conference_game': int(comp.get('conferenceCompetition', False)),
            }
    games = pd.DataFrame(rows.values())
    start = pd.to_datetime(games['start_utc'], utc=True)
    games['game_date_et'] = start.dt.tz_convert('America/New_York').dt.strftime('%Y-%m-%d')
    shifted = (start.dt.strftime('%Y-%m-%d') != games['game_date_et']).sum()

    crosswalk = team_crosswalk.assign(is_acc=team_crosswalk['is_acc'].astype(int))
    stats = {'unfinished_games_dropped': len(unfinished - set(rows)),
             'games_whose_date_moved_to_ET': int(shifted)}
    return {'teams': teams, 'team_crosswalk': crosswalk, 'ratings': ratings, 'games': games}, stats

def load(conn):
    """Upsert every table. Safe to run any number of times."""
    frames, stats = build_frames()
    conn.executescript(SCHEMA)
    with conn:
        for table, df in frames.items():
            cols = list(df.columns)
            sql = (f"INSERT OR REPLACE INTO {table} ({', '.join(cols)}) "
                   f"VALUES ({', '.join('?' * len(cols))})")
            conn.executemany(sql, df.astype(object).where(df.notna(), None).itertuples(index=False))
    return stats

def row_counts(conn):
    return {t: conn.execute(f'SELECT COUNT(*) FROM {t}').fetchone()[0]
            for t in ['teams', 'team_crosswalk', 'ratings', 'games']}

conn = sqlite3.connect(DB_PATH)
conn.execute('PRAGMA foreign_keys = ON')
load_stats = load(conn)

for name, sql in conn.execute("SELECT name, sql FROM sqlite_master WHERE type = 'table'"):
    print(sql, end='\n\n')
counts_first = row_counts(conn)
print('Row counts:', counts_first)
print('Load notes:', load_stats)


CREATE TABLE teams (
    torvik_name TEXT PRIMARY KEY,
    conf        TEXT NOT NULL
)

CREATE TABLE team_crosswalk (
    espn_id      TEXT PRIMARY KEY,
    espn_name    TEXT NOT NULL,
    espn_short   TEXT,
    torvik_name  TEXT REFERENCES teams(torvik_name),   -- NULL = not Division I, unrated
    match_method TEXT NOT NULL,
    is_acc       INTEGER NOT NULL
)

CREATE TABLE ratings (
    snapshot_date TEXT NOT NULL,
    torvik_name   TEXT NOT NULL REFERENCES teams(torvik_name),
    rank          INTEGER,
    wins          INTEGER,
    losses        INTEGER,
    adjoe         REAL,
    adjde         REAL,
    barthag       REAL,
    adjt          REAL,
    sos           REAL,
    ncsos         REAL,
    wab           REAL,
    PRIMARY KEY (snapshot_date, torvik_name)
)

CREATE TABLE games (
    game_id         TEXT PRIMARY KEY,
    game_date_et    TEXT NOT NULL,      -- local date in America/New_York
    start_utc       TEXT NOT NULL,      -- exactly as ESPN wrote it
    home_espn_id 

### 5. The rerun test

**TODO:** run your load a second time and show that every row count is unchanged. If the counts grow, the load is appending duplicates, and every average you compute later will be wrong in a way that is hard to notice.

In [5]:
# TODO

load(conn)
counts_second = row_counts(conn)
print('First load: ', counts_first)
print('Second load:', counts_second)
assert counts_first == counts_second, 'row counts changed: the load is appending duplicates'
print('Rerun test passed')

First load:  {'teams': 365, 'team_crosswalk': 110, 'ratings': 365, 'games': 95}
Second load: {'teams': 365, 'team_crosswalk': 110, 'ratings': 365, 'games': 95}
Rerun test passed


### 6. Decision log

**TODO:** list every decision so far that changed the data, with the number of rows each one affected. The crosswalk fixes, the time zone conversion, and the handling of unfinished or unrated games all belong here.

In [6]:
# TODO

unrated_ids = set(team_crosswalk.loc[team_crosswalk['torvik_name'].isna(), 'espn_id'])
games_vs_unrated = conn.execute(
    f"SELECT COUNT(*) FROM games WHERE home_espn_id IN ({','.join('?'*len(unrated_ids))}) "
    f"OR away_espn_id IN ({','.join('?'*len(unrated_ids))})",
    [*unrated_ids, *unrated_ids]).fetchone()[0] if unrated_ids else 0
state_rule = ((team_crosswalk['match_method'] == 'auto') &
              (team_crosswalk['espn_name'] != team_crosswalk['torvik_name'])).sum()
n_snapshots = len(list((RAW / 'torvik').glob('*.csv')))

decision_log = pd.DataFrame([
    ('Source', 'Torvik blocks requests from Colab (403), so each snapshot is downloaded in a browser and uploaded with the download date in the file name', n_snapshots * 365),
    ('Source', 'Deleted torvik_2026-11-17.csv: byte-identical to the Nov 20 download, so it was not a real snapshot', 365),
    ('Torvik', 'Dropped the duplicate rank.1 column; it equals rank on every row', 365 * n_snapshots),
    ('Torvik', 'Split record text like "6-0" into integer wins and losses', 365 * n_snapshots),
    ('Crosswalk', 'Matched on ESPN location (e.g. "Florida State") and keyed by ESPN team id, not the short display name', len(team_crosswalk)),
    ('Crosswalk', 'Automatic rule: ESPN "X State" -> Torvik "X St."', int(state_rule)),
    ('Crosswalk', 'Manual fixes: ' + ', '.join(f'{k} -> {v}' for k, v in MANUAL.items()), int(fixed.sum())),
    ('Crosswalk', f'Non-Division I opponents ({", ".join(still_miss)}) kept with no rating; their games count toward records but are left out of rating-based averages, SOS and Q6 favorites', games_vs_unrated),
    ('Games', 'Dropped games ESPN had not marked completed', load_stats['unfinished_games_dropped']),
    ('Games', 'Converted ESPN start times from UTC to America/New_York before taking the game date; these games moved to the previous day', load_stats['games_whose_date_moved_to_ET']),
    ('Load', 'Primary keys plus INSERT OR REPLACE so reruns update rows instead of appending; rerun test showed identical counts', sum(counts_first.values())),
], columns=['area', 'decision', 'rows_affected'])
decision_log.index = decision_log.index + 1

pd.set_option('display.max_colwidth', None)
print(f'{len(decision_log)} decisions logged')
decision_log


11 decisions logged


,area,decision,rows_affected
1,Source,"Torvik blocks requests from Colab (403), so each snapshot is downloaded in a browser and uploaded with the download date in the file name",365
2,Source,"Deleted torvik_2026-11-17.csv: byte-identical to the Nov 20 download, so it was not a real snapshot",365
3,Torvik,Dropped the duplicate rank.1 column; it equals rank on every row,365
4,Torvik,"Split record text like ""6-0"" into integer wins and losses",365
5,Crosswalk,"Matched on ESPN location (e.g. ""Florida State"") and keyed by ESPN team id, not the short display name",110
6,Crosswalk,"Automatic rule: ESPN ""X State"" -> Torvik ""X St.""",11
7,Crosswalk,"Manual fixes: NC State -> N.C. State, Miami -> Miami FL, Loyola Maryland -> Loyola MD, American University -> American, Texas A&M-Corpus Christi -> Texas A&M Corpus Chris",5
8,Crosswalk,"Non-Division I opponents (Adams State) kept with no rating; their games count toward records but are left out of rating-based averages, SOS and Q6 favorites",1
9,Games,Dropped games ESPN had not marked completed,7
10,Games,Converted ESPN start times from UTC to America/New_York before taking the game date; these games moved to the previous day,70


### 7. Self-check

In [7]:
setup = {
    'torvik_snapshots_saved': 1,      # TODO: an integer
    'scoreboard_dates_saved': 20,      # TODO: an integer
    'acc_teams_matched': 18,           # TODO: should be 18
    'games_table_rows': 95,            # TODO: an integer
    'rerun_counts_unchanged': True,     # TODO: True after the test in section 5
    'decisions_logged': 11,            # TODO: how many entries
    'repo_url': 'https://github.com/karen-tran08/ds2002-acc-reality-check',
    'notebook_runs_clean': True,        # TODO: True after Restart and Run All
}

for label, value in setup.items():
    ok = not (value is None or value is False
              or (isinstance(value, str) and value.startswith('TODO')))
    print('PASS' if ok else 'FAIL', '-', label, '=', value)

PASS - torvik_snapshots_saved = 1
PASS - scoreboard_dates_saved = 20
PASS - acc_teams_matched = 18
PASS - games_table_rows = 95
PASS - rerun_counts_unchanged = True
PASS - decisions_logged = 11
PASS - repo_url = https://github.com/karen-tran08/ds2002-acc-reality-check
PASS - notebook_runs_clean = True


### 8. Plan for the break

**a)** Question 6 compares ratings taken before each game with the result of the game, which only works if your team saves a new ratings snapshot regularly from now on. Who will take the snapshots during Thanksgiving week, and on which days?

**b)** What is the most likely thing to go wrong before November 30, and what is your team doing about it?

**a.** Karen takes the snapshots, since she owns the ratings CSV and crosswalk. The days are Sunday 11/22, Tuesday 11/24, Friday 11/27 and Sunday 11/29. That is two a week, the minimum the capstone asks for, with one on each side of Thanksgiving so every game that week has a recent rating from before tip-off. Each snapshot is a browser download of the Torvik CSV, uploaded as data/raw/torvik/torvik_YYYY-MM-DD.csv using the real download date and committed to the repo the same day. After each snapshot, we re-run the download cell so the ESPN scoreboards and summaries catch up too.

**b.** The most likely problem is a missed or mislabeled snapshot. Torvik blocks Colab, so every snapshot is a manual download that is easy to forget over the holiday. A missed day cannot be recovered, because Torvik only serves current ratings, and Q6 would have to use older ratings for those games. To mitigate this problem, we will have fixed days and phone reminders for each one, naming every file with the actual download date and checking it is not identical to the previous snapshot before committing, and committing each snapshot and acc.db to GitHub right away, since Colab wipes its storage when the session ends